# 03 — Approach 2: MobileNetV2 transfer learning

Use ImageNet pretrained MobileNetV2 with a new four-class head. The current helper freezes the backbone and trains the head for a lightweight baseline. Evaluate checkpoints on validation only during fitting.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = next((path for path in (Path.cwd(), *Path.cwd().parents) if (path / 'src').is_dir() and (path / 'data').is_dir()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError('Could not find the project root. Open this notebook from the project folder or one of its subfolders.')
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
import torch
from torch.utils.data import DataLoader
from src.dataset import DEFAULT_CLASSES, torch_datasets
from src.dl_model import build_model, train_model

DATA_ROOT = PROJECT_ROOT / 'data' / 'raw'
missing = [f'{split}/{label}' for split in ('train', 'test') for label in DEFAULT_CLASSES if not (DATA_ROOT / split / label).is_dir()]
if missing:
    raise FileNotFoundError(f"Missing orange dataset folders under {DATA_ROOT}: {', '.join(missing)}")

(train_ds, val_ds, test_ds), class_names = torch_datasets(DATA_ROOT)
train_loader = DataLoader(train_ds, batch_size=32, shuffle=True, num_workers=0)
val_loader = DataLoader(val_ds, batch_size=32, shuffle=False, num_workers=0)
model = build_model(num_classes=len(class_names), pretrained=True)
checkpoint_path = PROJECT_ROOT / 'models' / 'mobilenetv2_best.pth'
model, history = train_model(
    model,
    train_loader,
    val_loader,
    epochs=8,
    lr=1e-3,
    save_path=str(checkpoint_path),
)

history_df = pd.DataFrame(history)
display(history_df)
history_path = PROJECT_ROOT / 'data' / 'processed' / 'mobilenetv2_history.csv'
history_path.parent.mkdir(parents=True, exist_ok=True)
history_df.to_csv(history_path, index=False)
print('Best validation checkpoint saved to', checkpoint_path)
print('Training history saved to', history_path)


## Rationale and limitations

Transfer learning provides a compact baseline for the blackspot, canker, fresh, and grenning orange classes. A frozen backbone is fast but may underfit this domain; if validation learning curves justify it, unfreeze only the last blocks and tune with a small learning rate. Compare compute, latency, and generalization as well as quality. Pretraining does not guarantee robustness to new orchards, lighting, or cameras.